# B.O.S.S. — Test YOLOv8 su recordings

**Pipeline:**
1. Ricerca frame in tutte le cartelle disponibili
2. Caricamento `best.pt` + inferenza → frame annotati in `metrics/plots/annotated_frames/`
3. Distribuzione classi e confidenza
4. Preparazione Ground Truth (`boss-recordings1`)
5. Valutazione quantitativa contro GT (mAP, Precision, Recall)
6. Metriche per classe + grafici
7. Griglia campione frame + dashboard riepilogo

Tutti i grafici vengono salvati in `metrics/plots/`.

---
**Kaggle — cosa caricare:**
- **Model** `boss-yolo-checkpoint` → `best.pt`
- **Dataset** `boss-recordings` → cartella `recordings/` con dentro `frames/` e `frames (1)/`
- **Dataset** `boss-recordings1` → cartella con `train/images/`, `train/labels/`, `data.yaml`

In [ ]:
# ============================================================
# Cella 1 — Import librerie
# ============================================================
%pip install ultralytics pyyaml tqdm --quiet

import os
import shutil
import zipfile
import random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import yaml

import matplotlib.pyplot as plt

from tqdm import tqdm
from ultralytics import YOLO

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# Cella 2 — Configurazione (locale + Kaggle)
# ============================================================
import torch

IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

if IS_KAGGLE:
    KAGGLE_FRAME_ROOT = Path("/kaggle/input/datasets/lorenzoverdura/boss-recordings/recordings")
    KAGGLE_MODEL_DIR  = Path("/kaggle/input/models/lorenzoverdura/boss-yolo-checkpoint/pytorch/default/9")
    KAGGLE_GT_DIR     = Path("/kaggle/input/datasets/lorenzoverdura/boss-recordings1")
    BASE_DIR          = Path("/kaggle/working")
    MODEL_PATH        = KAGGLE_MODEL_DIR / "best.pt"
    RECORDINGS_ZIP    = None
    GT_ZIP_PATH       = None
else:
    BASE_DIR          = Path("/home/lorenzoverdura8/BOSS_CODE")
    MODEL_PATH        = BASE_DIR / "best.pt"
    RECORDINGS_ZIP    = BASE_DIR / "recordings.zip"
    GT_ZIP_PATH       = BASE_DIR / "BOSS_recordings.yolov8.zip"
    KAGGLE_FRAME_ROOT = None
    KAGGLE_GT_DIR     = None

# Output — struttura: metrics/plots/ e metrics/reports/
OUTPUT_DIR   = BASE_DIR / "metrics"
PLOTS_DIR    = OUTPUT_DIR / "plots"
REPORTS_DIR  = OUTPUT_DIR / "reports"
GT_EXTRACTED = BASE_DIR / "ground_truth_boss"

# Classi — ordine identico al training (boss_yolo_training.ipynb)
BOSS_CLASSES = [
    "Bike",                   # 0
    "Building",               # 1
    "Car",                    # 2
    "Person",                 # 3
    "Stairs",                 # 4
    "Traffic sign",           # 5
    "Electrical Pole",        # 6
    "Road",                   # 7
    "Motorcycle",             # 8
    "Dustbin",                # 9
    "Dog",                    # 10
    "Manhole",                # 11
    "Tree",                   # 12
    "Guard rail",             # 13
    "Pedestrian crosswalk",   # 14
    "Truck",                  # 15
    "Bus",                    # 16
    "Bench",                  # 17
    "Traffic Cone",           # 18
    "Fire hydrant",           # 19
    "Teraffic Barrel",        # 20
    "Plant Pot",              # 21
    "Electrical Box",         # 22
    "Chair",                  # 23
    "Bicycle Rack",           # 24
]
NUM_CLASSES      = len(BOSS_CLASSES)  # 25
IMG_SIZE         = 640
CONF_THRESHOLD   = 0.25
IOU_THRESHOLD    = 0.45
DEVICE           = "0" if torch.cuda.is_available() else "cpu"
MIN_SAMPLES_WARN = 10   # classi con < N campioni GT → avviso nel report

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
GT_EXTRACTED.mkdir(parents=True, exist_ok=True)

print(f"Ambiente:         {'Kaggle' if IS_KAGGLE else 'Locale'}")
print(f"MODEL_PATH:       {MODEL_PATH}  —  esiste: {Path(str(MODEL_PATH)).exists()}")
if KAGGLE_FRAME_ROOT:
    print(f"KAGGLE_FRAME_ROOT: {KAGGLE_FRAME_ROOT}  —  esiste: {KAGGLE_FRAME_ROOT.exists()}")
if KAGGLE_GT_DIR:
    print(f"KAGGLE_GT_DIR:    {KAGGLE_GT_DIR}  —  esiste: {KAGGLE_GT_DIR.exists()}")
print(f"OUTPUT_DIR:       {OUTPUT_DIR}")
print(f"PLOTS_DIR:        {PLOTS_DIR}")
print(f"REPORTS_DIR:      {REPORTS_DIR}")
print(f"DEVICE:           {DEVICE}")
print(f"Classi ({NUM_CLASSES}): {BOSS_CLASSES}")

In [ ]:
# ============================================================
# Cella 3 — Estrazione ZIP e ricerca frame
# ============================================================

if IS_KAGGLE:
    if not KAGGLE_FRAME_ROOT.exists():
        raise FileNotFoundError(f"Cartella frame non trovata: {KAGGLE_FRAME_ROOT}")
    frame_dirs = sorted([p for p in KAGGLE_FRAME_ROOT.iterdir() if p.is_dir()])
    all_frames = []
    for fd in frame_dirs:
        imgs = sorted(fd.glob("*.jpg")) + sorted(fd.glob("*.png"))
        all_frames.extend(imgs)
        print(f"  {fd.name}: {len(imgs)} frame")
    print(f"\nTotale frame trovati: {len(all_frames)}")
else:
    # In locale: estrai recordings.zip se necessario, poi cerca le cartelle frames
    recordings_root = BASE_DIR / "recordings"

    if not recordings_root.exists() or not any(recordings_root.iterdir()):
        if not RECORDINGS_ZIP.exists():
            raise FileNotFoundError(f"ZIP non trovato: {RECORDINGS_ZIP}")
        print(f"Estrazione {RECORDINGS_ZIP} ...")
        with zipfile.ZipFile(RECORDINGS_ZIP, "r") as z:
            z.extractall(BASE_DIR)
        print(f"  -> {recordings_root}")
    else:
        print(f"OK  recordings/ già presente")

    frame_dirs = sorted([
        p for p in recordings_root.rglob("frames")
        if p.is_dir()
    ])

    all_frames = []
    for fd in frame_dirs:
        imgs = sorted(fd.glob("*.jpg")) + sorted(fd.glob("*.png"))
        all_frames.extend(imgs)
        print(f"  {fd.relative_to(BASE_DIR)}: {len(imgs)} frame")

    print(f"\nTotale frame trovati: {len(all_frames)}")

if len(all_frames) == 0:
    raise FileNotFoundError("Nessun frame trovato.")

In [ ]:
# ============================================================
# Cella 4 — Caricamento modello e inferenza (Fase A)
# ============================================================
trained_model = YOLO(str(MODEL_PATH))
print(f"Modello caricato: {MODEL_PATH}")

inference_results = trained_model.predict(
    source   = [str(f) for f in all_frames],
    conf     = CONF_THRESHOLD,
    iou      = IOU_THRESHOLD,
    imgsz    = IMG_SIZE,
    device   = DEVICE,
    save     = True,
    save_txt = True,
    project  = str(PLOTS_DIR),
    name     = "annotated_frames",
    exist_ok = True,
    stream   = True,
)

all_predictions = []
PREDICT_SAVE_DIR = None

for result in inference_results:
    if PREDICT_SAVE_DIR is None:
        PREDICT_SAVE_DIR = Path(result.save_dir)

    frame_name = Path(result.path).name
    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        continue

    for i in range(len(boxes)):
        cls_id = int(boxes.cls[i].item())
        conf   = float(boxes.conf[i].item())
        xyxy   = boxes.xyxy[i].cpu().numpy()
        all_predictions.append({
            "frame":      frame_name,
            "class_id":   cls_id,
            "class_name": BOSS_CLASSES[cls_id] if cls_id < NUM_CLASSES else "unknown",
            "confidence": conf,
            "x1": float(xyxy[0]), "y1": float(xyxy[1]),
            "x2": float(xyxy[2]), "y2": float(xyxy[3]),
        })

df_preds = pd.DataFrame(all_predictions)

print(f"\nCartella frame annotati: {PREDICT_SAVE_DIR}")
print(f"Totale predizioni:       {len(df_preds)}")
print(f"Frame con predizioni:    {df_preds['frame'].nunique() if not df_preds.empty else 0}")
if not df_preds.empty:
    print(df_preds.head(10).to_string(index=False))

df_preds.to_csv(REPORTS_DIR / "predictions.csv", index=False)
print(f"\nSalvato: {REPORTS_DIR}/predictions.csv")

In [ ]:
# ============================================================
# Cella 5 — Distribuzione classi e confidenza
# ============================================================
if df_preds.empty:
    print("Nessuna predizione — grafici saltati.")
else:
    class_counts = (
        df_preds["class_name"]
        .value_counts()
        .reindex(BOSS_CLASSES, fill_value=0)
    )
    fig, ax = plt.subplots(figsize=(16, 5))
    colors = list(plt.cm.tab20.colors) * ((NUM_CLASSES // 20) + 1)
    ax.bar(class_counts.index, class_counts.values, color=colors[:NUM_CLASSES])
    ax.set_title("Distribuzione classi rilevate — recordings", fontsize=14)
    ax.set_xlabel("Classe")
    ax.set_ylabel("Numero rilevamenti")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "class_distribution.png", dpi=150)
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(df_preds["confidence"], bins=50, color="steelblue", edgecolor="white")
    ax.axvline(CONF_THRESHOLD, color="red", linestyle="--",
               label=f"soglia={CONF_THRESHOLD}")
    ax.set_title("Distribuzione confidenza predizioni", fontsize=13)
    ax.set_xlabel("Confidenza")
    ax.set_ylabel("Frequenza")
    ax.legend()
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "confidence_distribution.png", dpi=150)
    plt.show()

In [ ]:
# ============================================================
# Cella 6 — Preparazione Ground Truth (BOSS_recordings.yolov8)
# ============================================================

if IS_KAGGLE:
    # File già estratti nel dataset — usa i path diretti (read-only)
    GT_SRC_IMGS    = KAGGLE_GT_DIR / "train" / "images"
    GT_SRC_LABELS  = KAGGLE_GT_DIR / "train" / "labels"
    gt_yaml_in_path = KAGGLE_GT_DIR / "data.yaml"
    print(f"OK  GT già presente in {KAGGLE_GT_DIR}")
else:
    # Locale: estrai lo ZIP se necessario
    gt_yaml_in_path = GT_EXTRACTED / "data.yaml"
    if not gt_yaml_in_path.exists():
        if not GT_ZIP_PATH.exists():
            raise FileNotFoundError(f"ZIP ground truth non trovato: {GT_ZIP_PATH}")
        print(f"Estrazione {GT_ZIP_PATH} ...")
        with zipfile.ZipFile(GT_ZIP_PATH, "r") as z:
            z.extractall(GT_EXTRACTED)
        print(f"  -> {GT_EXTRACTED}")
    else:
        print(f"OK  ground_truth_boss/ già presente")
    GT_SRC_IMGS   = GT_EXTRACTED / "test" / "images"
    GT_SRC_LABELS = GT_EXTRACTED / "test" / "labels"

# Legge il data.yaml di Roboflow per ottenere l'ordine classi originale
with open(gt_yaml_in_path) as f:
    gt_yaml_in = yaml.safe_load(f)
RF_CLASSES = gt_yaml_in["names"]
print(f"Classi Roboflow ({len(RF_CLASSES)}): {RF_CLASSES}")

# Mappa RF_id → BOSS_id (case-insensitive)
RF_TO_BOSS_ID = {}
for rf_id, rf_name in enumerate(RF_CLASSES):
    for boss_id, boss_name in enumerate(BOSS_CLASSES):
        if rf_name.lower().strip() == boss_name.lower():
            RF_TO_BOSS_ID[rf_id] = boss_id
            break
unmapped = [RF_CLASSES[i] for i in range(len(RF_CLASSES)) if i not in RF_TO_BOSS_ID]
print(f"Mappa RF→BOSS: {RF_TO_BOSS_ID}")
print(f"Classi Roboflow scartate (non in BOSS_CLASSES): {unmapped}")


def remap_labels(src_label_dir, dst_label_dir, id_map):
    """Copia i file .txt YOLO rimappando i class ID da RF a BOSS."""
    src = Path(src_label_dir)
    dst = Path(dst_label_dir)
    dst.mkdir(parents=True, exist_ok=True)
    remapped, dropped = 0, 0
    for lf in src.glob("*.txt"):
        out_lines = []
        for line in lf.read_text().strip().splitlines():
            parts = line.strip().split()
            if not parts:
                continue
            rf_id = int(parts[0])
            if rf_id not in id_map:
                dropped += 1
                continue
            parts[0] = str(id_map[rf_id])
            out_lines.append(" ".join(parts))
            remapped += 1
        (dst / lf.name).write_text("\n".join(out_lines))
    print(f"  Annotazioni rimappate: {remapped} | Righe scartate: {dropped}")


# Copia immagini e label rimappate in ground_truth_boss/test/ (cartella scrivibile)
GT_TEST_IMGS   = GT_EXTRACTED / "test" / "images"
GT_TEST_LABELS = GT_EXTRACTED / "test" / "labels"
GT_TEST_IMGS.mkdir(parents=True, exist_ok=True)

img_count = 0
for ext in ("*.jpg", "*.jpeg", "*.png"):
    for img in GT_SRC_IMGS.glob(ext):
        shutil.copy(img, GT_TEST_IMGS / img.name)
        img_count += 1
print(f"Immagini copiate in test/images: {img_count}")

remap_labels(GT_SRC_LABELS, GT_TEST_LABELS, RF_TO_BOSS_ID)

# Scrive data_boss.yaml per YOLOv8 val()
gt_data_yaml = {
    "path":  str(GT_EXTRACTED),
    "train": "",
    "val":   "",
    "test":  "test/images",
    "nc":    NUM_CLASSES,
    "names": BOSS_CLASSES,
}
gt_yaml_out = GT_EXTRACTED / "data_boss.yaml"
with open(gt_yaml_out, "w") as f:
    yaml.dump(gt_data_yaml, f, default_flow_style=False, allow_unicode=True)
print(f"\ndata_boss.yaml scritto in: {gt_yaml_out}")

In [ ]:
# ============================================================
# Cella 7 — Valutazione quantitativa contro Ground Truth (Fase B)
# ============================================================
gt_val_results = trained_model.val(
    data     = str(gt_yaml_out),
    split    = "test",
    imgsz    = IMG_SIZE,
    conf     = 0.001,   # bassa per massimizzare recall nel calcolo mAP
    iou      = IOU_THRESHOLD,
    device   = DEVICE,
    plots    = True,
    project  = str(PLOTS_DIR),
    name     = "gt_eval",
    exist_ok = True,
)

map50     = float(gt_val_results.box.map50)
map5095   = float(gt_val_results.box.map)
precision = float(gt_val_results.box.mp)
recall    = float(gt_val_results.box.mr)
f1_score  = 2 * (precision * recall) / (precision + recall + 1e-9)

GT_EVAL_SAVE_DIR = Path(gt_val_results.save_dir)

print("\n=== Metriche contro Ground Truth ====")
print(f"mAP@0.5:       {map50:.4f}")
print(f"mAP@0.5:0.95:  {map5095:.4f}")
print(f"Precision:     {precision:.4f}")
print(f"Recall:        {recall:.4f}")
print(f"F1 Score:      {f1_score:.4f}")
print(f"Plot/CM in:    {GT_EVAL_SAVE_DIR}")

In [ ]:
# ============================================================
# Cella 8 — Metriche per classe + grafici GT
# ============================================================

# Conta campioni GT per classe (dai label rimappati in GT_TEST_LABELS)
gt_samples_per_class = {i: 0 for i in range(NUM_CLASSES)}
for lf in GT_TEST_LABELS.rglob("*.txt"):
    for line in lf.read_text().strip().splitlines():
        parts = line.strip().split()
        if parts:
            cid = int(parts[0])
            if cid in gt_samples_per_class:
                gt_samples_per_class[cid] += 1

gt_class_indices = gt_val_results.box.ap_class_index.astype(int)
gt_class_names   = [BOSS_CLASSES[i] for i in gt_class_indices]

gt_per_class_ap50   = gt_val_results.box.ap50
gt_per_class_ap5095 = gt_val_results.box.ap
gt_per_class_p      = gt_val_results.box.p
gt_per_class_r      = gt_val_results.box.r
gt_per_class_f1     = 2 * (gt_per_class_p * gt_per_class_r) / (gt_per_class_p + gt_per_class_r + 1e-9)

df_metrics = pd.DataFrame({
    "Classe":      gt_class_names,
    "GT_campioni": [gt_samples_per_class[i] for i in gt_class_indices],
    "AP@0.5":      gt_per_class_ap50,
    "AP@0.5:0.95": gt_per_class_ap5095,
    "Precision":   gt_per_class_p,
    "Recall":      gt_per_class_r,
    "F1":          gt_per_class_f1,
})
summary_row = pd.DataFrame([{
    "Classe":      "MEDIA",
    "GT_campioni": sum(gt_samples_per_class.values()),
    "AP@0.5":      df_metrics["AP@0.5"].mean(),
    "AP@0.5:0.95": df_metrics["AP@0.5:0.95"].mean(),
    "Precision":   df_metrics["Precision"].mean(),
    "Recall":      df_metrics["Recall"].mean(),
    "F1":          df_metrics["F1"].mean(),
}])
df_metrics = pd.concat([df_metrics, summary_row], ignore_index=True)

float_cols = ["AP@0.5", "AP@0.5:0.95", "Precision", "Recall", "F1"]
df_display = df_metrics.copy()
for col in float_cols:
    df_display[col] = df_display[col].apply(lambda x: f"{x:.4f}")
print("=== Metriche per Classe ===")
print(df_display.to_string(index=False))

low_sample_classes = [
    BOSS_CLASSES[i] for i, c in gt_samples_per_class.items()
    if 0 < c < MIN_SAMPLES_WARN
]
if low_sample_classes:
    print(f"\n⚠ Classi con meno di {MIN_SAMPLES_WARN} campioni GT (metriche poco affidabili):")
    for cls in low_sample_classes:
        print(f"  {cls}: {gt_samples_per_class[BOSS_CLASSES.index(cls)]} campioni")

df_metrics.to_csv(REPORTS_DIR / "metrics_per_class.csv", index=False)

df_plot = df_metrics[df_metrics["Classe"] != "MEDIA"].copy()

# AP per classe
x = np.arange(len(df_plot))
width = 0.35
fig, ax = plt.subplots(figsize=(16, 6))
bars1 = ax.bar(x - width/2, df_plot["AP@0.5"],      width, label="AP@0.5",      color="steelblue")
bars2 = ax.bar(x + width/2, df_plot["AP@0.5:0.95"], width, label="AP@0.5:0.95", color="coral")
for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f"{h:.2f}",
            ha="center", va="bottom", fontsize=8)
ax.set_title("AP per classe — Ground Truth", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(df_plot["Classe"], rotation=45, ha="right")
ax.set_ylabel("AP")
ax.set_ylim(0, 1.1)
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "ap_per_class.png", dpi=150)
plt.show()

# Precision vs Recall scatter
fig, ax = plt.subplots(figsize=(10, 7))
for _, row in df_plot.iterrows():
    ax.scatter(row["Recall"], row["Precision"], s=100, zorder=5)
    ax.annotate(row["Classe"], (row["Recall"], row["Precision"]),
                textcoords="offset points", xytext=(5, 3), fontsize=8)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision vs Recall per classe — Ground Truth")
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "precision_recall_scatter.png", dpi=150)
plt.show()

# Confusion matrix generata da YOLOv8 → copiata in plots/
cm_path = GT_EVAL_SAVE_DIR / "confusion_matrix_normalized.png"
if cm_path.exists():
    img_cm = Image.open(cm_path)
    plt.figure(figsize=(12, 10))
    plt.imshow(img_cm)
    plt.axis("off")
    plt.title("Confusion Matrix Normalizzata — Ground Truth")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "confusion_matrix_normalized.png", dpi=150)
    plt.show()
else:
    print(f"Confusion matrix non trovata in {cm_path}")

In [ ]:
# ============================================================
# Cella 8b — Esportazione report TXT e JSON
# ============================================================
import json as _json
from datetime import datetime as _dt

total_gt_imgs   = len(list(GT_TEST_IMGS.glob("*.jpg"))) + len(list(GT_TEST_IMGS.glob("*.png")))
total_gt_annots = sum(gt_samples_per_class.values())
classes_no_gt   = [BOSS_CLASSES[i] for i, c in gt_samples_per_class.items() if c == 0]
classes_low_gt  = [BOSS_CLASSES[i] for i, c in gt_samples_per_class.items() if 0 < c < MIN_SAMPLES_WARN]

# ── TXT ──────────────────────────────────────────────────────
txt_lines = [
    "=" * 62,
    "B.O.S.S. — Risultati Test YOLOv8",
    f"Data:    {_dt.now().strftime('%Y-%m-%d %H:%M')}",
    f"Modello: {MODEL_PATH}",
    f"Device:  {DEVICE}  |  ImgSize: {IMG_SIZE}  |  IoU: {IOU_THRESHOLD}",
    "=" * 62,
    "",
    "── Dataset ──────────────────────────────────────────────",
    f"  Sensore:                       Intel RealSense D435",
    f"  Frame recordings (inferenza):  {len(all_frames)}",
    f"  Immagini GT (valutazione):     {total_gt_imgs}",
    f"  Annotazioni GT totali:         {total_gt_annots}",
    f"  Classi senza GT:               {classes_no_gt if classes_no_gt else 'nessuna'}",
    f"  Classi con < {MIN_SAMPLES_WARN} campioni GT:   {classes_low_gt if classes_low_gt else 'nessuna'}",
    "",
    "── Metriche aggregate ────────────────────────────────────",
    f"  mAP@0.5:       {map50:.4f}",
    f"  mAP@0.5:0.95:  {map5095:.4f}",
    f"  Precision:     {precision:.4f}",
    f"  Recall:        {recall:.4f}",
    f"  F1 Score:      {f1_score:.4f}",
    "",
    "── Metriche per classe ───────────────────────────────────",
    f"  {'Classe':<25} {'AP@0.5':>7} {'P':>7} {'R':>7} {'F1':>7} {'GT':>6}",
    "  " + "-" * 58,
]
for _, row in df_metrics.iterrows():
    n = row["GT_campioni"]
    gt_str = "-" if row["Classe"] == "MEDIA" else str(int(n))
    warn = "  ⚠" if isinstance(n, (int, float)) and 0 < n < MIN_SAMPLES_WARN else ""
    txt_lines.append(
        f"  {row['Classe']:<25} {float(row['AP@0.5']):>7.4f} "
        f"{float(row['Precision']):>7.4f} {float(row['Recall']):>7.4f} "
        f"{float(row['F1']):>7.4f} {gt_str:>6}{warn}"
    )
txt_lines += [
    "",
    "NOTA: Dataset piccolo (RealSense D435). Metriche per classi",
    f"con < {MIN_SAMPLES_WARN} campioni GT hanno alta varianza e scarsa affidabilità.",
    "=" * 62,
]

txt_path = REPORTS_DIR / "metrics_summary.txt"
txt_path.write_text("\n".join(txt_lines), encoding="utf-8")
print(f"Salvato: {txt_path}")
print("\n".join(txt_lines))

# ── JSON ─────────────────────────────────────────────────────
per_class_list = []
for _, row in df_metrics[df_metrics["Classe"] != "MEDIA"].iterrows():
    cls_name = row["Classe"]
    cls_idx  = BOSS_CLASSES.index(cls_name)
    n        = gt_samples_per_class[cls_idx]
    per_class_list.append({
        "class_id":    cls_idx,
        "class_name":  cls_name,
        "gt_samples":  n,
        "low_samples": 0 < n < MIN_SAMPLES_WARN,
        "ap50":        round(float(row["AP@0.5"]), 4),
        "ap50_95":     round(float(row["AP@0.5:0.95"]), 4),
        "precision":   round(float(row["Precision"]), 4),
        "recall":      round(float(row["Recall"]), 4),
        "f1":          round(float(row["F1"]), 4),
    })

json_data = {
    "meta": {
        "date":       _dt.now().strftime("%Y-%m-%d %H:%M"),
        "model":      str(MODEL_PATH),
        "device":     DEVICE,
        "img_size":   IMG_SIZE,
        "iou":        IOU_THRESHOLD,
        "conf_infer": CONF_THRESHOLD,
        "sensor":     "Intel RealSense D435",
        "note": (
            f"Dataset piccolo. Metriche per classi con < {MIN_SAMPLES_WARN} campioni GT "
            "hanno alta varianza e sono da interpretare con cautela."
        ),
    },
    "aggregate": {
        "map50":     round(map50, 4),
        "map50_95":  round(map5095, 4),
        "precision": round(precision, 4),
        "recall":    round(recall, 4),
        "f1":        round(f1_score, 4),
    },
    "dataset_stats": {
        "inference_frames": len(all_frames),
        "gt_images":        total_gt_imgs,
        "gt_annotations":   total_gt_annots,
        "classes_no_gt":    classes_no_gt,
        "classes_low_gt":   classes_low_gt,
    },
    "per_class": per_class_list,
}

json_path = REPORTS_DIR / "metrics_boss.json"
json_path.write_text(_json.dumps(json_data, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nSalvato: {json_path}")

In [ ]:
# ============================================================
# Cella 9 — Griglia campione frame annotati
# ============================================================
N_SAMPLES = 9

if PREDICT_SAVE_DIR is None or not PREDICT_SAVE_DIR.exists():
    print(f"Cartella inferenza non disponibile: {PREDICT_SAVE_DIR}")
    annotated_imgs = []
else:
    annotated_imgs = (
        list(PREDICT_SAVE_DIR.glob("*.jpg")) +
        list(PREDICT_SAVE_DIR.glob("*.png"))
    )

if len(annotated_imgs) == 0:
    print("Nessun frame annotato disponibile per la griglia.")
else:
    sample = random.sample(annotated_imgs, min(N_SAMPLES, len(annotated_imgs)))
    grid_size = int(np.ceil(np.sqrt(len(sample))))

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(5 * grid_size, 4 * grid_size))
    axes_flat = axes.flat if hasattr(axes, "flat") else [axes]

    for ax, img_path in zip(axes_flat, sample):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(img_path.name, fontsize=7)
        ax.axis("off")
    for ax in list(axes_flat)[len(sample):]:
        ax.axis("off")

    plt.suptitle("Campione frame recordings — predizioni B.O.S.S.", fontsize=14)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "sample_predictions.png", dpi=150)
    plt.show()

In [ ]:
# ============================================================
# Cella 10 — Dashboard riepilogo finale
# ============================================================
fig = plt.figure(figsize=(16, 10))
fig.suptitle("B.O.S.S. — Riepilogo Test su recordings", fontsize=16, fontweight="bold")

ax1 = fig.add_subplot(2, 3, 1)
metrics_summary_vals = {
    "mAP@0.5":      map50,
    "mAP@0.5:0.95": map5095,
    "Precision":    precision,
    "Recall":       recall,
    "F1 Score":     f1_score,
}
colors_gauge = ["#2196F3", "#1976D2", "#4CAF50", "#FF9800", "#9C27B0"]
bars_h = ax1.barh(list(metrics_summary_vals.keys()), list(metrics_summary_vals.values()), color=colors_gauge)
for bar, val in zip(bars_h, metrics_summary_vals.values()):
    ax1.text(min(val + 0.02, 0.95), bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", va="center", fontsize=10, fontweight="bold")
ax1.set_xlim(0, 1.1)
ax1.set_title("Metriche Aggregate vs GT", fontsize=11)
ax1.axvline(0.5, color="red", linestyle="--", alpha=0.5, label="soglia 0.5")
ax1.legend(fontsize=8)

ax2 = fig.add_subplot(2, 3, 2)
ax2.barh(df_plot["Classe"], df_plot["AP@0.5"], color="steelblue")
ax2.set_title("AP@0.5 per Classe", fontsize=11)
ax2.set_xlim(0, 1)
ax2.axvline(map50, color="red", linestyle="--", alpha=0.7, label=f"media={map50:.2f}")
ax2.legend(fontsize=8)

ax3 = fig.add_subplot(2, 3, 3)
ax3.barh(df_plot["Classe"], df_plot["F1"], color="coral")
ax3.set_title("F1 Score per Classe", fontsize=11)
ax3.set_xlim(0, 1)
ax3.axvline(f1_score, color="red", linestyle="--", alpha=0.7, label=f"media={f1_score:.2f}")
ax3.legend(fontsize=8)

ax4 = fig.add_subplot(2, 3, 4)
scatter_colors = list(plt.cm.tab20.colors) * ((len(df_plot) // 20) + 1)
for i, (_, row) in enumerate(df_plot.iterrows()):
    ax4.scatter(row["Recall"], row["Precision"],
                color=scatter_colors[i], s=100, zorder=5)
    ax4.annotate(row["Classe"], (row["Recall"], row["Precision"]),
                 textcoords="offset points", xytext=(5, 3), fontsize=7)
ax4.set_xlabel("Recall")
ax4.set_ylabel("Precision")
ax4.set_title("Precision vs Recall per Classe", fontsize=11)
ax4.set_xlim(0, 1.05)
ax4.set_ylim(0, 1.05)
ax4.grid(True, alpha=0.3)

ax5 = fig.add_subplot(2, 3, 5)
if not df_preds.empty:
    class_counts_test = df_preds["class_name"].value_counts()
    ax5.pie(class_counts_test.values,
            labels=class_counts_test.index,
            autopct="%1.1f%%",
            startangle=140,
            textprops={"fontsize": 7})
    ax5.set_title("Distribuzione classi — Frame recordings", fontsize=11)
else:
    ax5.text(0.5, 0.5, "Nessuna predizione", ha="center", va="center")
    ax5.axis("off")

ax6 = fig.add_subplot(2, 3, 6)
ax6.axis("off")
frames_count = len(all_frames)
pred_count   = len(df_preds)
gt_img_count = len(list(GT_TEST_IMGS.glob("*.jpg"))) + len(list(GT_TEST_IMGS.glob("*.png")))
info_text = (
    f"Modello:        {Path(str(MODEL_PATH)).name}\n"
    f"Sensore:        RealSense D435\n"
    f"Classi:         {NUM_CLASSES}\n"
    f"Immagine:       {IMG_SIZE}x{IMG_SIZE}\n"
    f"Conf threshold: {CONF_THRESHOLD}\n"
    f"IoU threshold:  {IOU_THRESHOLD}\n\n"
    f"Frame recordings:  {frames_count}\n"
    f"Pred. totali:      {pred_count}\n"
    f"Pred/frame:        {pred_count / max(frames_count, 1):.1f}\n"
    f"GT immagini test:  {gt_img_count}\n"
)
ax6.text(0.05, 0.95, info_text, transform=ax6.transAxes,
         fontsize=10, verticalalignment="top", fontfamily="monospace",
         bbox=dict(boxstyle="round", facecolor="#f0f0f0", alpha=0.8))
ax6.set_title("Configurazione", fontsize=11)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nOutput completo in: {OUTPUT_DIR}")
print(f"  {PLOTS_DIR.name}/")
for p in sorted(PLOTS_DIR.glob("*.png")):
    print(f"    {p.name}")
print(f"  {REPORTS_DIR.name}/")
for p in sorted(REPORTS_DIR.iterdir()):
    print(f"    {p.name}")